# RPS Predictor — 5-fold CV: Overfitting Analysis

Plots train vs val MSE curves across all folds to identify the epoch where
overfitting begins.

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

CV_RESULTS = "../results/rps_cv/cv_results.json"

with open(CV_RESULTS) as f:
    cv = json.load(f)

folds = sorted(cv.keys())
n_folds = len(folds)
n_epochs = max(len(cv[k]) for k in folds)
print(f"Folds: {n_folds}  |  Epochs recorded: {n_epochs}")

In [ ]:
def extract(fold_key, metric):
    return np.array([e[metric] for e in cv[fold_key]])

def cv_curve(metric):
    """Return (epochs, mean_curve, std_curve) across all folds."""
    curves = np.array([extract(k, metric) for k in folds])
    return np.arange(1, curves.shape[1] + 1), curves.mean(axis=0), curves.std(axis=0)

epochs, train_mean, train_std = cv_curve("train_mse")
epochs, val_mean,   val_std   = cv_curve("val_mse")
epochs, mae_mean,   mae_std   = cv_curve("val_mae_clip")

print(f"Best mean val_mse:  {val_mean.min():.4f} at epoch {val_mean.argmin()+1}")
print(f"Best mean MAE/clip: {mae_mean.min():.4f} at epoch {mae_mean.argmin()+1}")

In [ ]:
# ── Main learning-curve plot ───────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = plt.cm.tab10(np.linspace(0, 0.5, n_folds))

# Left: all individual fold curves + mean
ax = axes[0]
for i, k in enumerate(folds):
    t = extract(k, "train_mse")
    v = extract(k, "val_mse")
    ep = np.arange(1, len(t) + 1)
    ax.plot(ep, t, color=colors[i], linewidth=0.8, alpha=0.4)
    ax.plot(ep, v, color=colors[i], linewidth=1.2, alpha=0.8, label=f'fold {i}')
ax.plot(epochs, train_mean, 'k--', linewidth=1.5, label='train mean')
ax.plot(epochs, val_mean,   'k-',  linewidth=2.0, label='val mean')
ax.set_xlabel('Epoch')
ax.set_ylabel('MSE')
ax.set_title('Train (dashed) vs Val (solid) MSE — all folds')
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

# Right: mean ± std band
ax = axes[1]
ax.plot(epochs, train_mean, 'tab:blue',  linewidth=1.5, label='train mean')
ax.fill_between(epochs,
                train_mean - train_std, train_mean + train_std,
                alpha=0.15, color='tab:blue')
ax.plot(epochs, val_mean,   'tab:orange', linewidth=1.5, label='val mean')
ax.fill_between(epochs,
                val_mean - val_std, val_mean + val_std,
                alpha=0.15, color='tab:orange')
best_ep = int(val_mean.argmin()) + 1
ax.axvline(best_ep, color='red', linestyle='--', linewidth=1.0,
           label=f'best val epoch={best_ep}')
ax.set_xlabel('Epoch')
ax.set_ylabel('MSE')
ax.set_title('Mean ± std across 5 folds')
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

plt.suptitle('5-fold CV learning curves — simple_conv RPS predictor', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# ── Generalisation gap: val_mse - train_mse ───────────────────────────────
fig, ax = plt.subplots(figsize=(9, 4))

for i, k in enumerate(folds):
    gap = extract(k, "val_mse") - extract(k, "train_mse")
    ep  = np.arange(1, len(gap) + 1)
    ax.plot(ep, gap, color=colors[i], linewidth=0.9, alpha=0.6, label=f'fold {i}')

gap_mean = val_mean - train_mean
ax.plot(epochs, gap_mean, 'k-', linewidth=2.0, label='mean gap')
ax.axhline(0, color='gray', linestyle='--', linewidth=0.8)
ax.set_xlabel('Epoch')
ax.set_ylabel('val_mse − train_mse')
ax.set_title('Generalisation gap (positive = overfitting)')
ax.legend(fontsize=8, ncol=3)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ── MAE/clip curve ─────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(epochs, mae_mean, 'tab:green', linewidth=1.5, label='val MAE/clip mean')
ax.fill_between(epochs, mae_mean - mae_std, mae_mean + mae_std,
                alpha=0.2, color='tab:green')
best_mae_ep = int(mae_mean.argmin()) + 1
ax.axvline(best_mae_ep, color='red', linestyle='--', linewidth=1.0,
           label=f'best MAE epoch={best_mae_ep}')
ax.set_xlabel('Epoch')
ax.set_ylabel('MAE / clip (RPS)')
ax.set_title('Val MAE per clip — mean ± std across folds')
ax.legend(fontsize=9)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ── Summary table: metrics at selected epoch checkpoints ──────────────────
checkpoints = [10, 20, 30, 40, 50, 60, 70, 80, 90, 100]
checkpoints = [e for e in checkpoints if e <= n_epochs]

print(f"{'Epoch':>6}  {'TrainMSE':>10}  {'ValMSE':>10}  {'Gap':>8}  {'MAE/clip':>9}")
print("-" * 50)
for ep in checkpoints:
    i = ep - 1
    if i < len(train_mean):
        print(f"{ep:6d}  {train_mean[i]:10.4f}  {val_mean[i]:10.4f}  "
              f"{val_mean[i]-train_mean[i]:8.4f}  {mae_mean[i]:9.4f}")

print()
print(f"Best val_mse = {val_mean.min():.4f} ± {val_std[val_mean.argmin()]:.4f}  at epoch {best_ep}")
print(f"Best MAE/clip = {mae_mean.min():.4f} ± {mae_std[mae_mean.argmin()]:.4f}  at epoch {best_mae_ep}")